# Fine-Tuning Llama-3 8B on WhatsApp Data

This notebook:
1. Loads configuration from config.yaml
2. Loads train/validation datasets
3. Initializes the model with LoRA adapters
4. Trains with validation monitoring
5. Saves the fine-tuned model

In [1]:
import sys
import os
import yaml
from datasets import load_dataset

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', '..', 'src')))

from kaya_chatbot.trainer import KayaTrainer

c:\Users\guga\Desktop\KayaChatBot\kaya_chatbot_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
# 1. Load Configuration
with open(r"C:\Users\guga\Desktop\KayaChatBot\config.yaml", 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Extract settings
MODEL_ID = config['model']['model_id']
MAX_SEQ_LENGTH = config['model']['max_seq_length']
OUTPUT_DIR = config['training']['output_dir']
TRAIN_FILE = r"C:\Users\guga\Desktop\KayaChatBot\data\train_data.jsonl"
VAL_FILE = r"C:\Users\guga\Desktop\KayaChatBot\data\val_data.jsonl"

print(f"✓ Configuration loaded")
print(f"  Model: {MODEL_ID}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Output directory: {OUTPUT_DIR}")

In [ ]:
# 2. Load Datasets
train_dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
val_dataset = load_dataset("json", data_files=VAL_FILE, split="train")

print(f"✓ Loaded {len(train_dataset)} training examples")
print(f"✓ Loaded {len(val_dataset)} validation examples")

In [ ]:
# 3. Initialize Trainer with LoRA config
lora_config = {
    "r": config['model']['lora_r'],
    "alpha": config['model']['lora_alpha'],
    "dropout": config['model']['lora_dropout'],
    "target_modules": config['model']['target_modules']
}

trainer_wrapper = KayaTrainer(
    model_id=MODEL_ID, 
    max_seq_length=MAX_SEQ_LENGTH,
    lora_config=lora_config
)

print("Loading model... (this may take a few minutes)")
trainer_wrapper.load_model()

In [ ]:
# 4. Train the Model
training_config = {
    "max_steps": config['training']['max_steps'],
    "per_device_train_batch_size": config['training']['per_device_train_batch_size'],
    "gradient_accumulation_steps": config['training']['gradient_accumulation_steps'],
    "learning_rate": config['training']['learning_rate'],
    "warmup_steps": config['training']['warmup_steps'],
    "weight_decay": config['training']['weight_decay'],
    "optim": config['training']['optim'],
    "lr_scheduler_type": config['training']['lr_scheduler_type'],
    "logging_steps": config['training']['logging_steps'],
    "save_steps": config['training']['save_steps'],
    "eval_steps": config['training']['eval_steps'],
    "seed": config['training']['seed'],
}

print(f"\n🚀 Starting training with {training_config['max_steps']} steps...")
trainer, stats = trainer_wrapper.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    output_dir=OUTPUT_DIR,
    training_config=training_config
)

print("\n✅ Training completed!")
print(f"Final stats: {stats}")

In [ ]:
# 5. Save Model
trainer_wrapper.save_model(OUTPUT_DIR)
print(f"\n💾 Model and adapters saved to: {OUTPUT_DIR}")

In [ ]:
# 5. Inference Test (Optional)
from unsloth import FastLanguageModel

model, tokenizer = trainer_wrapper.model, trainer_wrapper.tokenizer
FastLanguageModel.for_inference(model)

prompt = """
Gil João: O que acham desta música?
Peter: """

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])